# LoCoMotif Step-by-Step Mid Study: From Self-Similarity to Time-Warped Motif Sets

This notebook is a mid-study notebook for understanding and preparing a LoCoMotif experiment with real thesis data. It is not the final benchmark notebook and it does not claim final motif results.

The purpose is to make the LoCoMotif workflow transparent:

- load processed crypto data from the thesis project,
- select a manageable BTCUSDT slice,
- build multivariate feature vectors,
- normalize them,
- construct a Self-Similarity Matrix (SSM),
- demonstrate the intuition of time-warped local paths,
- prepare clean output folders and schema files for later official LoCoMotif extraction.

All motif-like paths produced here are educational diagnostics only. They are not official LoCoMotif motif sets.


## Why LoCoMotif?

Financial patterns may repeat with different speed, duration, and local distortion. A sell-off pattern, volatility burst, liquidity shock, or rebound can happen quickly in one period and more slowly in another.

Matrix Profile methods are strong for fixed-window motifs: they ask whether subsequences of a fixed length are similar. This is useful, but it can miss repetitions whose duration changes.

LoCoMotif is useful when motif candidates may be variable length, time-warped, and multivariate. Instead of requiring two segments to line up point by point at the same speed, it searches for locally consistent paths through a similarity matrix.


In [ ]:
from pathlib import Path
import os
import json
import warnings
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)

print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"NumPy: {np.__version__}")
print(f"pandas: {pd.__version__}")
print(f"scikit-learn: {sklearn.__version__}")


In [ ]:
PROJECT_ROOT = Path(r"C:\Users\learn\OneDrive\Desktop\Final Masters Thesis")

primary_data_root = PROJECT_ROOT / "data"
fallback_data_root = PROJECT_ROOT / "final_dataset"

def has_parquet_files(root: Path) -> bool:
    try:
        return root.exists() and any(root.rglob("*.parquet"))
    except Exception:
        return False

DATA_ROOT = primary_data_root if has_parquet_files(primary_data_root) else fallback_data_root

CONFIG = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "NOTEBOOK_DIR": PROJECT_ROOT / "notebooks" / "Locomotif Mid study",
    "DATA_ROOT": DATA_ROOT,
    "RESULTS_DIR": PROJECT_ROOT / "reports" / "results" / "locomotif_mid_study",
    "FIGURES_DIR": PROJECT_ROOT / "reports" / "results" / "locomotif_mid_study" / "figures",
    "TABLES_DIR": PROJECT_ROOT / "reports" / "results" / "locomotif_mid_study" / "tables",
    "CONFIG_DIR": PROJECT_ROOT / "reports" / "results" / "locomotif_mid_study" / "config",
    "INTERMEDIATE_DIR": PROJECT_ROOT / "reports" / "results" / "locomotif_mid_study" / "intermediate",
    "ASSET": "BTCUSDT",
    "OPTIONAL_ASSET_2": "ETHUSDT",
    "START_DATE": "2025-01-01",
    "END_DATE": "2025-01-07",
    "RESAMPLE_RULE": "5min",
    "FEATURE_COLUMNS_CANDIDATES": [
        "log_return",
        "rolling_volatility",
        "volatility_60m",
        "realized_volatility_60m",
        "hl_range",
        "volume_zscore",
        "volume",
        "close",
    ],
    "SSM_MAX_POINTS": 800,
    "LOCOMOTIF_L_MIN": 12,
    "LOCOMOTIF_L_MAX": 72,
    "LOCOMOTIF_RHO": 0.8,
    "LOCOMOTIF_NU": 0.25,
}

for key in ["NOTEBOOK_DIR", "FIGURES_DIR", "TABLES_DIR", "CONFIG_DIR", "INTERMEDIATE_DIR"]:
    CONFIG[key].mkdir(parents=True, exist_ok=True)

def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {k: json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    return value

config_json_path = CONFIG["CONFIG_DIR"] / "locomotif_mid_study_config.json"
with config_json_path.open("w", encoding="utf-8") as f:
    json.dump(json_ready(CONFIG), f, indent=2)

print("Created or verified output directories:")
for key in ["NOTEBOOK_DIR", "FIGURES_DIR", "TABLES_DIR", "CONFIG_DIR", "INTERMEDIATE_DIR"]:
    print(f"- {key}: {CONFIG[key]}")
print(f"- DATA_ROOT selected: {CONFIG['DATA_ROOT']}")
print(f"\nSaved configuration: {config_json_path}")

## Step 1: Locate Real Thesis Data

The notebook searches recursively inside the project `data` folder for likely processed parquet files. It prints every BTC/ETH candidate it finds so the data choice is auditable.

The primary asset is BTCUSDT. ETHUSDT is treated as an optional comparison asset if it exists.


In [ ]:
from datetime import datetime

DATA_ROOT = CONFIG["DATA_ROOT"]
patterns = ["BTCUSDT*.parquet", "ETHUSDT*.parquet", "*BTC*.parquet", "*ETH*.parquet"]

candidate_paths = []
for pattern in patterns:
    candidate_paths.extend(DATA_ROOT.rglob(pattern))

candidate_paths = sorted({p.resolve() for p in candidate_paths if p.is_file()})

def guess_asset(path):
    name = path.name.upper()
    path_text = str(path).upper()
    if "BTCUSDT" in name or "BTCUSDT" in path_text:
        return "BTCUSDT"
    if "ETHUSDT" in name or "ETHUSDT" in path_text:
        return "ETHUSDT"
    if "BTC" in name or "BTC" in path_text:
        return "BTC"
    if "ETH" in name or "ETH" in path_text:
        return "ETH"
    return "UNKNOWN"

candidate_records = []
for path in candidate_paths:
    stat = path.stat()
    candidate_records.append(
        {
            "asset_guess": guess_asset(path),
            "file_path": str(path),
            "file_size_mb": round(stat.st_size / (1024 ** 2), 3),
            "modified_time": pd.Timestamp(stat.st_mtime, unit="s"),
        }
    )

candidate_df = pd.DataFrame(candidate_records)
if not candidate_df.empty:
    candidate_df = candidate_df.sort_values(["asset_guess", "file_path"]).reset_index(drop=True)

print(f"Searched folder: {DATA_ROOT}")
print(f"Candidate parquet files found: {len(candidate_df)}")
display(candidate_df)


In [ ]:
def choose_asset_file(candidate_df, asset):
    if candidate_df.empty:
        raise FileNotFoundError(
            f"No candidate parquet files were found under {CONFIG['DATA_ROOT']}. "
            "Expected files matching BTCUSDT*.parquet, ETHUSDT*.parquet, *BTC*.parquet, or *ETH*.parquet."
        )

    asset_upper = asset.upper()
    exact_mask = candidate_df["asset_guess"].str.upper().eq(asset_upper)
    path_mask = candidate_df["file_path"].str.upper().str.contains(asset_upper, regex=False)
    matches = candidate_df.loc[exact_mask | path_mask].copy()

    if matches.empty:
        raise FileNotFoundError(
            f"Could not find {asset_upper} parquet data under {CONFIG['DATA_ROOT']}. "
            "Review the candidate dataframe above or check that processed BTCUSDT data exists."
        )

    matches["processed_rank"] = matches["file_path"].str.lower().str.contains("processed").astype(int)
    matches = matches.sort_values(["processed_rank", "modified_time", "file_size_mb"], ascending=[False, False, False])
    return Path(matches.iloc[0]["file_path"])

def maybe_choose_asset_file(candidate_df, asset):
    try:
        return choose_asset_file(candidate_df, asset)
    except FileNotFoundError:
        return None

btc_path = choose_asset_file(candidate_df, CONFIG["ASSET"])
eth_path = maybe_choose_asset_file(candidate_df, CONFIG["OPTIONAL_ASSET_2"])

print(f"Selected BTCUSDT file: {btc_path}")
if eth_path is not None:
    print(f"Optional ETHUSDT file found: {eth_path}")
else:
    print("Optional ETHUSDT file was not found. Continuing with BTCUSDT only.")

btc_df = pd.read_parquet(btc_path)

def find_datetime_column(df):
    candidates = ["timestamp", "open_time", "datetime", "date", "time", "close_time"]
    lower_map = {str(c).lower(): c for c in df.columns}
    for name in candidates:
        if name in lower_map:
            return lower_map[name]
    return None

def convert_datetime_values(values):
    series = pd.Series(values)
    numeric = pd.to_numeric(series, errors="coerce")
    numeric_share = numeric.notna().mean()

    if numeric_share > 0.8:
        median_abs = numeric.dropna().abs().median()
        if median_abs > 1e17:
            unit = "ns"
        elif median_abs > 1e14:
            unit = "us"
        elif median_abs > 1e11:
            unit = "ms"
        else:
            unit = "s"
        return pd.to_datetime(numeric, unit=unit, errors="coerce")

    return pd.to_datetime(series, errors="coerce")

def ensure_datetime_index(df):
    df = df.copy()

    if isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors="coerce")
    else:
        datetime_col = find_datetime_column(df)
        if datetime_col is None:
            raise ValueError(
                "The loaded dataframe does not have a DatetimeIndex and no timestamp/date/time column was found. "
                "Expected one of: timestamp, open_time, datetime, date, time, close_time."
            )
        converted = convert_datetime_values(df[datetime_col])
        df = df.loc[converted.notna()].copy()
        df.index = pd.DatetimeIndex(converted.loc[converted.notna()])
        if datetime_col in df.columns:
            df = df.drop(columns=[datetime_col])

    if df.index.tz is not None:
        df.index = df.index.tz_convert(None)

    df = df.loc[df.index.notna()].sort_index()
    df = df[~df.index.duplicated(keep="last")]
    return df

btc_df = ensure_datetime_index(btc_df)

print(f"Loaded shape: {btc_df.shape}")
print(f"Index type: {type(btc_df.index)}")
print(f"Date range: {btc_df.index.min()} -> {btc_df.index.max()}")
print("\nColumns:")
print(list(btc_df.columns))

display(btc_df.head())

relevant_for_missing = [c for c in CONFIG["FEATURE_COLUMNS_CANDIDATES"] if c in btc_df.columns]
for required in ["open", "high", "low", "close", "volume"]:
    if required in btc_df.columns and required not in relevant_for_missing:
        relevant_for_missing.append(required)

if relevant_for_missing:
    missing_summary = btc_df[relevant_for_missing].isna().sum().rename("missing_count").to_frame()
    missing_summary["missing_pct"] = 100 * missing_summary["missing_count"] / len(btc_df)
    display(missing_summary)
else:
    print("No standard feature/OHLCV columns were found for missing-value summary.")


## Step 2: Select a Manageable Time Slice

The Self-Similarity Matrix is an `n x n` object. If the selected time slice contains 100,000 timestamps, the matrix would contain 10 billion pairwise entries, which is not appropriate for an explanatory notebook.

We therefore begin with a small BTCUSDT time slice and resample it to a manageable resolution before building the SSM.


In [ ]:
def find_column(df, name):
    lower_map = {str(c).lower(): c for c in df.columns}
    return lower_map.get(name.lower())

def resample_ohlcv_and_numeric(df, rule):
    agg = {}
    used = set()
    ohlcv_rules = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
    }

    for standard_name, method in ohlcv_rules.items():
        col = find_column(df, standard_name)
        if col is not None:
            agg[col] = method
            used.add(col)

    numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in used]
    for col in numeric_cols:
        agg[col] = "mean"

    if not agg:
        raise ValueError("No numeric columns were available for resampling.")

    return df.resample(rule).agg(agg)

start_date = pd.Timestamp(CONFIG["START_DATE"])
end_date = pd.Timestamp(CONFIG["END_DATE"])

slice_df = btc_df.loc[start_date:end_date].copy()
if slice_df.empty:
    raise ValueError(
        f"No BTCUSDT rows were found between {CONFIG['START_DATE']} and {CONFIG['END_DATE']}. "
        f"Available date range is {btc_df.index.min()} -> {btc_df.index.max()}."
    )

shape_before = slice_df.shape
resampled_df = resample_ohlcv_and_numeric(slice_df, CONFIG["RESAMPLE_RULE"])

close_col = find_column(resampled_df, "close")
if close_col is not None:
    resampled_df = resampled_df.dropna(subset=[close_col])
else:
    print("Warning: no close column found after resampling. Later price plots will be skipped.")

shape_after = resampled_df.shape

print(f"Slice period: {CONFIG['START_DATE']} -> {CONFIG['END_DATE']}")
print(f"Shape before resampling: {shape_before}")
print(f"Shape after {CONFIG['RESAMPLE_RULE']} resampling: {shape_after}")
print(f"Resampled date range: {resampled_df.index.min()} -> {resampled_df.index.max()}")
display(resampled_df.head())


## Step 3: Build Thesis-Style Multivariate Features

LoCoMotif can work with multivariate time series. For this experiment, every timestamp is represented as a vector of market-state features.

The notebook first reuses existing processed features when they are available. If a required feature is missing, it builds a simple transparent version from OHLCV data:

- `log_return`: log price change,
- `rolling_volatility`: rolling standard deviation of log returns,
- `hl_range`: high-low range relative to close,
- `volume_zscore`: standardized volume against a rolling local baseline.


In [ ]:
work_df = resampled_df.copy()

close_col = find_column(work_df, "close")
high_col = find_column(work_df, "high")
low_col = find_column(work_df, "low")
volume_col = find_column(work_df, "volume")

if close_col is None:
    raise ValueError("A close column is required to build return-based features.")

def rule_to_minutes(rule):
    try:
        return pd.Timedelta(rule).total_seconds() / 60
    except Exception:
        return 5.0

resample_minutes = max(rule_to_minutes(CONFIG["RESAMPLE_RULE"]), 1.0)
window_60m = max(2, int(round(60 / resample_minutes)))
volume_window = max(window_60m, 24)

if "log_return" not in work_df.columns:
    work_df["log_return"] = np.log(work_df[close_col]).diff()

if "pct_return" not in work_df.columns:
    work_df["pct_return"] = work_df[close_col].pct_change()

if "hl_range" not in work_df.columns and high_col is not None and low_col is not None:
    work_df["hl_range"] = (work_df[high_col] - work_df[low_col]) / work_df[close_col]

if "rolling_volatility" not in work_df.columns:
    work_df["rolling_volatility"] = work_df["log_return"].rolling(window_60m, min_periods=max(2, window_60m // 2)).std()

if "volatility_60m" not in work_df.columns:
    work_df["volatility_60m"] = work_df["log_return"].rolling(window_60m, min_periods=max(2, window_60m // 2)).std()

if "volume_zscore" not in work_df.columns and volume_col is not None:
    rolling_mean = work_df[volume_col].rolling(volume_window, min_periods=max(2, volume_window // 2)).mean()
    rolling_std = work_df[volume_col].rolling(volume_window, min_periods=max(2, volume_window // 2)).std()
    work_df["volume_zscore"] = (work_df[volume_col] - rolling_mean) / rolling_std.replace(0, np.nan)

selected_features = []
if "log_return" in work_df.columns:
    selected_features.append("log_return")

for candidate in ["rolling_volatility", "volatility_60m", "realized_volatility_60m"]:
    if candidate in work_df.columns:
        selected_features.append(candidate)
        break

if "hl_range" in work_df.columns:
    selected_features.append("hl_range")

if "volume_zscore" in work_df.columns:
    selected_features.append("volume_zscore")

selected_features = list(dict.fromkeys(selected_features))
if len(selected_features) < 2:
    raise ValueError(
        f"Only {len(selected_features)} usable feature(s) were available: {selected_features}. "
        "At least two features are recommended for this multivariate LoCoMotif preparation notebook."
    )

# Robust cleaning for sparse features (e.g., daily data or short slices)
raw_feature_df = work_df[selected_features].replace([np.inf, -np.inf], np.nan).copy()
coverage = raw_feature_df.notna().mean().sort_values(ascending=False)

# Keep features with enough observed data; fallback to best available two
keep_features = [c for c in raw_feature_df.columns if coverage[c] >= 0.20]
if len(keep_features) < 2:
    keep_features = list(coverage.index[: min(2, len(coverage))])

feature_df = raw_feature_df[keep_features].copy()

# Fill internal gaps for stability in exploratory notebook workflow
feature_df = feature_df.interpolate(method="time", limit_direction="both")
feature_df = feature_df.dropna()

if feature_df.empty:
    raise ValueError(
        "Feature dataframe is empty after cleaning. "
        f"rows_in_work_df={len(work_df)}, selected_features={selected_features}, "
        f"coverage={coverage.to_dict()}. "
        "Likely wrong timeframe file selected (e.g., 1d instead of intraday)."
    )

selected_features = keep_features

feature_path = CONFIG["INTERMEDIATE_DIR"] / f"{CONFIG['ASSET']}_locomotif_features.parquet"
feature_df.to_parquet(feature_path)

print(f"Selected features: {selected_features}")
print(f"Feature shape: {feature_df.shape}")
print(f"Saved feature parquet: {feature_path}")
display(feature_df.head())


## Step 4: Normalize Features

The SSM compares vectors across time. If one feature has a much larger numeric scale than another, it can dominate the distance calculation.

For multivariate LoCoMotif preparation, every timestamp becomes a vector:

`x_i = [return_i, volatility_i, range_i, volume_zscore_i]`

We standardize each feature column so that distances are driven by relative movement rather than raw units.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(feature_df.values)
scaled_df = pd.DataFrame(X_scaled, index=feature_df.index, columns=feature_df.columns)

scaled_feature_path = CONFIG["INTERMEDIATE_DIR"] / f"{CONFIG['ASSET']}_locomotif_scaled_features.parquet"
scaled_df.to_parquet(scaled_feature_path)
print(f"Saved scaled feature parquet: {scaled_feature_path}")

feature_figure_paths = []
for col in selected_features:
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(scaled_df.index, scaled_df[col], linewidth=0.9)
    ax.set_title(f"{CONFIG['ASSET']} scaled feature: {col}")
    ax.set_xlabel("Time")
    ax.set_ylabel("Standardized value")
    ax.grid(True, alpha=0.25)
    fig.autofmt_xdate()
    fig.tight_layout()

    fig_path = CONFIG["FIGURES_DIR"] / f"feature_{col}.png"
    fig.savefig(fig_path, dpi=150)
    feature_figure_paths.append(fig_path)
    print(f"Saved figure: {fig_path}")
    plt.show()


## Step 5: Build the Self-Similarity Matrix

The Self-Similarity Matrix compares every timestamp with every other timestamp. For this notebook, similarity is defined as:

`S(i, j) = exp(-||x_i - x_j||^2)`

where `x_i` and `x_j` are multivariate feature vectors at two timestamps.


In [ ]:
n_ssm = min(CONFIG["SSM_MAX_POINTS"], len(scaled_df))
if n_ssm < 2:
    raise ValueError("At least two scaled rows are required to build an SSM.")

ssm_df = scaled_df.iloc[:n_ssm].copy()
dist_squared = pairwise_distances(ssm_df.values, metric="sqeuclidean")
S = np.exp(-dist_squared)

ssm_path = CONFIG["INTERMEDIATE_DIR"] / f"{CONFIG['ASSET']}_SSM_small.npy"
np.save(ssm_path, S)
print(f"SSM shape: {S.shape}")
print(f"Saved SSM: {ssm_path}")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(S, origin="lower", aspect="auto", cmap="viridis")
ax.set_title("Self-Similarity Matrix: BTCUSDT multivariate features")
ax.set_xlabel("Time index j")
ax.set_ylabel("Time index i")
fig.colorbar(im, ax=ax, label="Similarity")
fig.tight_layout()

ssm_fig_path = CONFIG["FIGURES_DIR"] / "01_self_similarity_matrix.png"
fig.savefig(ssm_fig_path, dpi=150)
print(f"Saved figure: {ssm_fig_path}")
plt.show()


## Interpreting the Self-Similarity Matrix

The main diagonal is the self-match: each timestamp is maximally similar to itself.

Off-diagonal bright diagonal stripes indicate repeated subsequences. A straight stripe suggests that two segments evolve at approximately the same speed.

Curved, broken, or uneven stripes suggest time distortion, imperfect repetition, local acceleration, or local deceleration. This is the visual motivation for time-warped motif discovery.


## Step 6: Admissible Warping Steps

LoCoMotif uses admissible steps through the SSM to trace locally consistent similarity paths:

`A = {(1, 1), (1, 2), (2, 1)}`

- `(1, 1)` means both segments progress at the same speed.
- `(1, 2)` means the second segment progresses faster locally.
- `(2, 1)` means the first segment progresses faster locally.

This is the key time-warping idea: two repeated patterns do not need to have exactly the same duration or speed.


In [ ]:
toy_n = 100
x = np.arange(10, 90)
straight_y = x.copy()
warped_y = 10 + (x - 10) + 10 * np.sin((x - 10) / 80 * np.pi)
warped_y = np.clip(warped_y, 0, toy_n - 1)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(np.zeros((toy_n, toy_n)), origin="lower", cmap="Greys", alpha=0.15)
ax.plot(x, straight_y, label="Straight same-speed path", linewidth=2)
ax.plot(x, warped_y, label="Curved adaptive path", linewidth=2)
ax.set_title("Straight versus locally warped path")
ax.set_xlabel("Segment B index")
ax.set_ylabel("Segment A index")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.25)
fig.tight_layout()

path_demo_fig = CONFIG["FIGURES_DIR"] / "02_straight_vs_warped_path.png"
fig.savefig(path_demo_fig, dpi=150)
print(f"Saved figure: {path_demo_fig}")
plt.show()


## Step 7: Educational LoCo Dynamic Programming Demo

This section is not the official optimized LoCoMotif implementation.

It is a small educational dynamic-programming demonstration that shows how cumulative similarity paths can be formed from the SSM. It helps explain the mechanics before connecting the notebook to the real `dtai-locomotif` package.


In [ ]:
A = [(1, 1), (1, 2), (2, 1)]
tau = float(np.quantile(S, CONFIG["LOCOMOTIF_RHO"]))
delta_a = 2 * tau
delta_m = 0.5

n = S.shape[0]
D = np.zeros_like(S, dtype=np.float64)

print(f"Running educational DP on SSM with shape {S.shape}")
print(f"tau = {tau:.6f}, delta_a = {delta_a:.6f}, delta_m = {delta_m:.3f}")

for i in range(n):
    if n >= 300 and i % 100 == 0:
        print(f"Processed row {i}/{n}")
    for j in range(n):
        prev_values = []
        for di, dj in A:
            pi, pj = i - di, j - dj
            if pi >= 0 and pj >= 0:
                prev_values.append(D[pi, pj])
        max_prev = max(prev_values) if prev_values else 0.0

        if S[i, j] >= tau:
            D[i, j] = max_prev + S[i, j]
        else:
            D[i, j] = max(0.0, delta_m * max_prev - delta_a)

d_path = CONFIG["INTERMEDIATE_DIR"] / f"{CONFIG['ASSET']}_educational_D_matrix.npy"
np.save(d_path, D)
print(f"Saved educational cumulative similarity matrix: {d_path}")

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(D, origin="lower", aspect="auto", cmap="magma")
ax.set_title("Educational cumulative similarity matrix D")
ax.set_xlabel("Time index j")
ax.set_ylabel("Time index i")
fig.colorbar(im, ax=ax, label="Cumulative similarity")
fig.tight_layout()

d_fig_path = CONFIG["FIGURES_DIR"] / "03_educational_cumulative_similarity_D.png"
fig.savefig(d_fig_path, dpi=150)
print(f"Saved figure: {d_fig_path}")
plt.show()


`S` tells us where individual points are similar.

`D` tells us where strong paths of accumulated similarity end.

A high value in `D` is therefore not just one similar timestamp pair. It is evidence that a locally consistent path of similar timestamp pairs has accumulated up to that location.


## Step 8: Backtracking One Strong Path

The main diagonal of the SSM is the trivial self-match. To make the educational example more meaningful, the next cell excludes a band around the main diagonal before searching for a strong path endpoint.

The recovered path maps one time segment to another similar segment while allowing local stretching or compression.


In [ ]:
def backtrack_path(D, start_i, start_j, admissible_steps):
    path = []
    i, j = int(start_i), int(start_j)

    while i >= 0 and j >= 0 and D[i, j] > 0:
        path.append((i, j))
        candidates = []
        for di, dj in admissible_steps:
            pi, pj = i - di, j - dj
            if pi >= 0 and pj >= 0:
                candidates.append((D[pi, pj], pi, pj))

        if not candidates:
            break

        best_value, best_i, best_j = max(candidates, key=lambda item: item[0])
        if best_value <= 0:
            break
        i, j = best_i, best_j

    path.reverse()
    return path

I, J = np.indices(D.shape)
diagonal_exclusion = np.abs(I - J) < CONFIG["LOCOMOTIF_L_MIN"]
masked_D = np.where(diagonal_exclusion, -np.inf, D)

if np.isfinite(masked_D).any() and np.nanmax(masked_D) > 0:
    end_i, end_j = np.unravel_index(np.nanargmax(masked_D), masked_D.shape)
    recovered_path = backtrack_path(D, end_i, end_j, A)
    educational_path_found = len(recovered_path) > 1
else:
    end_i, end_j = None, None
    recovered_path = []
    educational_path_found = False

print(f"Educational path found: {educational_path_found}")
if educational_path_found:
    print(f"Path length in matrix coordinates: {len(recovered_path)}")
    print(f"Endpoint: i={end_i}, j={end_j}, D={D[end_i, end_j]:.6f}")
else:
    print("No non-trivial educational path was found outside the diagonal exclusion band.")

if educational_path_found:
    path_arr = np.array(recovered_path)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(S, origin="lower", aspect="auto", cmap="viridis")
    ax.plot(path_arr[:, 1], path_arr[:, 0], color="red", linewidth=1.8, label="Backtracked path")
    ax.set_title("Backtracked educational warping path on SSM")
    ax.set_xlabel("Time index j")
    ax.set_ylabel("Time index i")
    ax.legend(loc="upper left")
    fig.colorbar(im, ax=ax, label="Similarity")
    fig.tight_layout()
    s_path_fig = CONFIG["FIGURES_DIR"] / "04_backtracked_warping_path_on_SSM.png"
    fig.savefig(s_path_fig, dpi=150)
    print(f"Saved figure: {s_path_fig}")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(D, origin="lower", aspect="auto", cmap="magma")
    ax.plot(path_arr[:, 1], path_arr[:, 0], color="cyan", linewidth=1.8, label="Backtracked path")
    ax.set_title("Backtracked educational warping path on D")
    ax.set_xlabel("Time index j")
    ax.set_ylabel("Time index i")
    ax.legend(loc="upper left")
    fig.colorbar(im, ax=ax, label="Cumulative similarity")
    fig.tight_layout()
    d_path_fig = CONFIG["FIGURES_DIR"] / "05_backtracked_warping_path_on_D.png"
    fig.savefig(d_path_fig, dpi=150)
    print(f"Saved figure: {d_path_fig}")
    plt.show()


## Step 9: Convert Path Projections into Two Aligned Segments

The path has two coordinate projections:

- the `i` coordinates identify one segment,
- the `j` coordinates identify the other segment.

Mapping those coordinates back to timestamps gives an interpretable pair of candidate regions. These are not final LoCoMotif motif sets. They are one educational example of how a local warping path connects two similar regions.


In [ ]:
def z_normalize(values):
    values = np.asarray(values, dtype=float)
    std = np.nanstd(values)
    if not np.isfinite(std) or std == 0:
        return values - np.nanmean(values)
    return (values - np.nanmean(values)) / std

segment_figure_paths = []

if not educational_path_found:
    print("Skipping segment projection plots because no educational path was found.")
else:
    path_arr = np.array(recovered_path)
    projection_1 = path_arr[:, 0]
    projection_2 = path_arr[:, 1]

    times_1 = ssm_df.index[projection_1]
    times_2 = ssm_df.index[projection_2]

    print(f"Segment A time range: {times_1.min()} -> {times_1.max()}")
    print(f"Segment B time range: {times_2.min()} -> {times_2.max()}")

    close_col = find_column(work_df, "close")
    if close_col is not None:
        close_aligned = work_df.reindex(scaled_df.index)[close_col].iloc[:n_ssm]
        close_a = close_aligned.iloc[projection_1].to_numpy()
        close_b = close_aligned.iloc[projection_2].to_numpy()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(np.linspace(0, 1, len(close_a)), z_normalize(close_a), label="Segment A close", linewidth=1.8)
        ax.plot(np.linspace(0, 1, len(close_b)), z_normalize(close_b), label="Segment B close", linewidth=1.8)
        ax.set_title("Two time-warped candidate segments: normalized close")
        ax.set_xlabel("Normalized local path position")
        ax.set_ylabel("Z-normalized close")
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        close_fig = CONFIG["FIGURES_DIR"] / "06_two_time_warped_candidate_segments_close.png"
        fig.savefig(close_fig, dpi=150)
        segment_figure_paths.append(close_fig)
        print(f"Saved figure: {close_fig}")
        plt.show()
    else:
        print("No close column available, so close segment plot was skipped.")

    for feature in selected_features:
        feature_a = ssm_df[feature].iloc[projection_1].to_numpy()
        feature_b = ssm_df[feature].iloc[projection_2].to_numpy()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(np.linspace(0, 1, len(feature_a)), z_normalize(feature_a), label="Segment A", linewidth=1.8)
        ax.plot(np.linspace(0, 1, len(feature_b)), z_normalize(feature_b), label="Segment B", linewidth=1.8)
        ax.set_title(f"Educational segment comparison: {feature}")
        ax.set_xlabel("Normalized local path position")
        ax.set_ylabel("Z-normalized feature value")
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()

        safe_feature = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in feature)
        feature_segment_fig = CONFIG["FIGURES_DIR"] / f"07_segment_comparison_{safe_feature}.png"
        fig.savefig(feature_segment_fig, dpi=150)
        segment_figure_paths.append(feature_segment_fig)
        print(f"Saved figure: {feature_segment_fig}")
        plt.show()


## Step 10: Prepare Real LoCoMotif Integration Cell

The next cell intentionally does not fail if `locomotif` is unavailable. It only checks whether the package is importable in the current environment.

The official LoCoMotif run should be added after confirming the exact local package API used elsewhere in the thesis project.


In [ ]:
try:
    import locomotif
    real_locomotif_available = True
    print("locomotif package found")
except Exception as e:
    real_locomotif_available = False
    print("locomotif package not found yet")
    print("Later install or connect the local dtai-locomotif integration.")
    print(e)


## TODO for Real LoCoMotif Motif Extraction

1. Connect this notebook to the real `dtai-locomotif` package already used in the thesis project.
2. Run LoCoMotif on `scaled_df` with `l_min`, `l_max`, `rho`, and `nu`.
3. Parse returned motif sets.
4. Plot every motif set on price and feature channels.
5. Compare LoCoMotif motifs with Matrix Profile motifs on the same time slice.
6. Save motif-set tables with start/end timestamps, duration, fitness, and feature summary.


In [ ]:
motif_schema_columns = [
    "asset",
    "method",
    "motif_set_id",
    "occurrence_id",
    "start_time",
    "end_time",
    "duration_minutes",
    "l_min",
    "l_max",
    "rho",
    "nu",
    "fitness",
    "notes",
]

motif_results_schema = pd.DataFrame(columns=motif_schema_columns)
schema_csv_path = CONFIG["TABLES_DIR"] / "locomotif_results_schema.csv"
motif_results_schema.to_csv(schema_csv_path, index=False)

print(f"Saved empty motif results schema: {schema_csv_path}")
display(motif_results_schema)


## Step 11: Notebook Summary

What we achieved:

- Loaded real BTCUSDT data.
- Built multivariate features.
- Normalized the feature matrix.
- Built the SSM.
- Demonstrated local warping path extraction educationally.
- Prepared the notebook for real LoCoMotif motif-set extraction.

What remains:

- Run official LoCoMotif.
- Extract motif sets.
- Compare against Matrix Profile.
- Evaluate stability across regimes.


In [ ]:
important_outputs = []

for name in [
    "config_json_path",
    "feature_path",
    "scaled_feature_path",
    "ssm_path",
    "d_path",
    "schema_csv_path",
    "ssm_fig_path",
    "path_demo_fig",
    "d_fig_path",
]:
    if name in globals():
        path = Path(globals()[name])
        if path.exists():
            important_outputs.append(str(path))

for collection_name in ["feature_figure_paths", "segment_figure_paths"]:
    for path in globals().get(collection_name, []):
        path = Path(path)
        if path.exists():
            important_outputs.append(str(path))

for name in ["s_path_fig", "d_path_fig"]:
    if name in globals():
        path = Path(globals()[name])
        if path.exists():
            important_outputs.append(str(path))

status = {
    "notebook_name": "01_locomotif_step_by_step_mid_study.ipynb",
    "asset": CONFIG["ASSET"],
    "start_date": CONFIG["START_DATE"],
    "end_date": CONFIG["END_DATE"],
    "resample_rule": CONFIG["RESAMPLE_RULE"],
    "selected_features": selected_features if "selected_features" in globals() else [],
    "n_rows_features": int(len(feature_df)) if "feature_df" in globals() else 0,
    "ssm_shape": list(S.shape) if "S" in globals() else None,
    "educational_path_found": bool(globals().get("educational_path_found", False)),
    "real_locomotif_available": bool(globals().get("real_locomotif_available", False)),
    "outputs_created": sorted(set(important_outputs)),
}

status_json_path = CONFIG["CONFIG_DIR"] / "status.json"
with status_json_path.open("w", encoding="utf-8") as f:
    json.dump(status, f, indent=2)

print(f"Saved status JSON: {status_json_path}")
print(json.dumps(status, indent=2))

NOTEBOOK_PATH = CONFIG["NOTEBOOK_DIR"] / "01_locomotif_step_by_step_mid_study.ipynb"
HTML_PATH = CONFIG["NOTEBOOK_DIR"] / "01_locomotif_step_by_step_mid_study.html"

try:
    import subprocess
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "jupyter",
            "nbconvert",
            "--to",
            "html",
            str(NOTEBOOK_PATH),
            "--output",
            HTML_PATH.name,
            "--output-dir",
            str(CONFIG["NOTEBOOK_DIR"]),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.returncode == 0 and HTML_PATH.exists():
        print(f"Exported lightweight HTML notebook: {HTML_PATH}")
    else:
        print("nbconvert HTML export was not available or did not complete.")
        print(result.stderr.strip() or result.stdout.strip())
except Exception as e:
    print("nbconvert HTML export was not available. Continuing without HTML export.")
    print(e)
